# Scalable Data Approach — Tile-then-Rollup

The naive engine recomputes each window feature from raw authorization records — a near-full scan **per feature, per run**. On huge Parquet auth tables that's the cost problem.

This notebook demonstrates the production approach:

> **Scan the source ONCE into daily per-entity tiles. Every window feature — and the 30→90 change — is then a cheap roll-up over those tiles.**

```
raw auths --(one scan)--> account_daily_tiles --(cheap rollup)--> denials_30d
                                               --(cheap rollup)--> denials_90d   (SAME tiles)
                                               --(cheap rollup)--> txn_count_7d
                                               --(cheap rollup)--> amount_sum_30d
```

We instrument a **source-scan counter** to prove the raw table is touched only once.

In [ ]:
import os, sys
import pandas as pd
sys.path.insert(0, os.path.abspath(".."))
from feature_store import FeatureSpec
from feature_store import tiles, rollup

auth = pd.read_csv("../data/authorizations.csv")
print(f"{len(auth):,} raw authorizations")

## 1 · Define the features (same specs as before)

In [ ]:
specs = [
  FeatureSpec(name="denials_30d", entity="account_id", timestamp_col="txn_ts", source="authorizations",
              aggregation="count", window_days=30, filter_expr="decision == 'DENY'", data_type="int", owner="da60046"),
  FeatureSpec(name="txn_count_7d", entity="account_id", timestamp_col="txn_ts", source="authorizations",
              aggregation="count", window_days=7, data_type="int", owner="da60046"),
  FeatureSpec(name="amount_sum_30d", entity="account_id", timestamp_col="txn_ts", source="authorizations",
              aggregation="sum", agg_field="amount", window_days=30, data_type="double", owner="da60046"),
  FeatureSpec(name="distinct_mcc_7d", entity="account_id", timestamp_col="txn_ts", source="authorizations",
              aggregation="count_distinct", agg_field="mcc", window_days=7, data_type="int", owner="da60046"),
]

## 2 · Build tiles — the ONE and only source scan
The tile builder derives exactly the daily base measures these specs need and computes them all in a **single pass** over the raw records. In production this is a daily Spark `GROUP BY` over one date partition, appended to an Iceberg tile table.

In [ ]:
tiles.reset_scan_counter()
tile_tbl = tiles.build_tiles(auth, specs)

print(f"Source scans: {tiles.SOURCE_SCANS}   (one pass serves ALL features)")
print(f"Raw rows {len(auth):,}  ->  daily tiles {len(tile_tbl):,}")
print("Tile columns:", [c for c in tile_tbl.columns if c not in ('account_id','date')])
tile_tbl.head()

## 3 · Every feature is a cheap roll-up — no new source scan
Watch the scan counter stay at **1** while we compute all four features off the tiles.

In [ ]:
features = {}
for s in specs:
    features[s.name] = rollup.rollup(s, tile_tbl)
    print(f"{s.name:16s} mean={features[s.name][s.name].mean():.2f}   source_scans={tiles.SOURCE_SCANS}")

## 4 · The 30 → 90 change — still zero source recompute
A different window is just a different roll-up over tiles that already exist. The scan counter does not move.

In [ ]:
denials_90d = FeatureSpec(name="denials_90d", entity="account_id", timestamp_col="txn_ts", source="authorizations",
                          aggregation="count", window_days=90, filter_expr="decision == 'DENY'", data_type="int", owner="da60046")
f90 = rollup.rollup(denials_90d, tile_tbl)
print(f"denials_90d mean={f90['denials_90d'].mean():.2f}")
print(f"TOTAL source scans for ALL features + the 30->90 change: {tiles.SOURCE_SCANS}")
print("Compare: the naive engine would have scanned the raw table once PER feature.")

## 5 · Steady state: incremental daily tiles
You never recompute history. Each day's job scans only that day's new records and appends one immutable tile per active account.

In [ ]:
auth["txn_ts"] = pd.to_datetime(auth["txn_ts"])
cutoff = auth["txn_ts"].dt.floor("D").max()
history = auth[auth["txn_ts"].dt.floor("D") < cutoff]
new_day = auth[auth["txn_ts"].dt.floor("D") == cutoff]

tiles.reset_scan_counter()
tile_hist = tiles.build_tiles(history, specs)
tile_all  = tiles.build_tiles_incremental(tile_hist, new_day, specs)
print(f"New day raw rows scanned: {len(new_day):,}  (vs full table {len(auth):,})")
print(f"Tiles after append: {len(tile_all):,}")
print("Steady-state daily cost is bounded by ONE day of data, not all history.")

## 6 · The distinct-count caveat
Daily distinct counts can't simply be summed (an MCC seen on two days would double-count). The tile stores the **daily set** of values and the roll-up takes the **union**, then counts.

For low-cardinality fields (like MCC, a few hundred codes) exact sets are fine. For high-cardinality distinct counts in production, store a **mergeable HyperLogLog sketch** per day and merge over the window (approximate, standard practice).

In [ ]:
dm = features["distinct_mcc_7d"]
print("distinct_mcc_7d via daily-set union over the window:")
dm.head(8)

---
### What this proves about scale

| | Naive engine | Tile + roll-up |
|---|---|---|
| Source scans for N features | **N** (one per feature) | **1** |
| 30 → 90 change | full recompute | roll-up over existing tiles (no source scan) |
| New window of an existing measure | full recompute | nearly free |
| Steady-state daily cost | re-scan history | one day of new data |
| Reads at feature time | huge raw table | tiny daily tile table |

**Production mapping**
- tile builder → daily Spark `GROUP BY` over one **date partition** of the Parquet auth table, appended to an **Iceberg** tile table
- roll-up → windowed sum/min/max/union over tiles (cheap)
- keep the **massive source auths as Parquet**; write the small **tiles + features as Iceberg** for incremental reads + time-travel
- **partition source by date**, project only needed columns → the one scan that happens is bounded